In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --progress-bar off
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install -qqq --no-deps {xformers} trl peft accelerate bitsandbytes triton --progress-bar off


import unsloth
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer
from datasets import load_dataset
import trl
print(trl.__version__) #0.24.0


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
0.24.0


#Task 1
## Load Llama-3.2 1B modell




In [ ]:
# Load model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Prepare model for PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)
print(model.print_trainable_parameters())

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2025.11.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
None


In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}
)

def apply_template(examples):
    messages = examples["conversations"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

dataset = load_dataset("mlabonne/FineTome-100k", split="train")
dataset = dataset.shuffle(seed=42)

# train/test
train_and_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_and_test["train"]
test_dataset  = train_and_test["test"]

train_dataset = train_dataset.map(apply_template, batched=True)
test_dataset  = test_dataset.map(apply_template, batched=True)

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## Training

In [ ]:

PATH = "/content/drive/MyDrive/Scalable ML/Lab2/checkpoints"
small_test = test_dataset.shuffle(seed=42).select(range(500))

trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset = small_test,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        #num_train_epochs=1,
        max_steps = 1610,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir= PATH,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        seed=0,
        report_to = "none",
        remove_unused_columns = False,
    ),
)

try:
  trainer.train(resume_from_checkpoint=True)

except ValueError:
  trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90,000 | Num Epochs = 1 | Total steps = 1,610
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1602,0.939900
1603,1.039900
1604,0.843900
1605,0.947700
1606,1.100200
1607,0.900400
1608,1.061800
1609,0.916000
1610,0.684200


## Evaluate

In [ ]:
metrics = trainer.evaluate()
print(metrics)

Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


{'eval_loss': 0.9217607378959656, 'eval_runtime': 120.2832, 'eval_samples_per_second': 4.157, 'eval_steps_per_second': 0.524, 'epoch': 0.2862222222222222}


## Inference

In [ ]:
# Load model for inference
model = FastLanguageModel.for_inference(model)

messages = [
    {"from": "human", "value": "Is Sweden a country?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True)

<|im_start|>user
Is Sweden a country?<|im_end|>
<|im_start|>assistant
Yes, Sweden is indeed a country. It is a sovereign state located in the northern part of Europe, with a population of 11.5 million people. It is a constitutional monarchy, meaning it is ruled by a king, but the actual power lies in the hands of the parliament and the prime minister. The official languages of Sweden are Swedish and English. The country's capital is Stockholm, and its largest city is Gothenburg.<|im_end|>


## Save the LoRA adapters

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1")

('/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1/tokenizer_config.json',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1/special_tokens_map.json',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1/chat_template.jinja',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1/tokenizer.json')

##Load the LoRA adapters we just saved for inference

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "/content/drive/MyDrive/Scalable ML/Lab2/lora_model_1", # YOUR MODEL YOU USED FOR TRAINING
      max_seq_length = max_seq_length,
      load_in_4bit=True,
      dtype=None,
  )
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "Is Sweden a country?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True)

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Unsloth 2025.11.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
Is Sweden a country?<|im_end|>
<|im_start|>assistant
Yes, Sweden is indeed a country. It is a sovereign state located in the northernmost region of the European continent, bordered by Norway to the west, Finland to the east, the Baltic Sea to the north, and the Gulf of Bothnia to the south. Sweden is one of the largest countries in the world, with a total area of 438,459 square kilometers (169,510 square miles), and a population of over 11 million people. It is the largest Nordic country and the third largest in Europe, after Russia and France. Sweden is a constitutional monarchy, with a parliamentary system of government, and a strong democratic tradition


##Save the merged model

In [ ]:
#model.save_pretrained_gguf("/content/drive/MyDrive/Scalable ML/Lab2/model_1", tokenizer, quantization_method = "q4_k_m",   maximum_memory_usage = 0.5) #Kraschar pga tar för mycket RAM
model.save_pretrained_merged("/content/drive/MyDrive/Scalable ML/Lab2/model_1_a", tokenizer, save_method="merged_16bit")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 1125.99it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:13<00:00, 133.73s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Scalable ML/Lab2/model_1`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages


##Convert to gguf
The converting of hf to gguf can be found in "convert.ipynb".

## Upload the model to HuggingFace

In [ ]:
!pip install -U "huggingface_hub"
from huggingface_hub import login
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.2/516.2 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.2 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.1.7 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import create_repo

create_repo(
    repo_id="lab2-model-gguf",
    repo_type="model",
    private=False
)

RepoUrl('https://huggingface.co/jacoblb/lab2-model-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='jacoblb/lab2-model-gguf')

In [ ]:
from huggingface_hub import upload_file

local_path = "/content/drive/MyDrive/Scalable ML/Lab2/model_1_a-q4_k_m.gguf"
repo_id = "jacoblb/lab2-model-gguf"
path_in_repo = "model_1_a-q4_k_m.gguf"

upload_file(
    path_or_fileobj=local_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ab2/model_1_a-q4_k_m.gguf:   3%|3         | 25.1MB /  808MB            

CommitInfo(commit_url='https://huggingface.co/jacoblb/lab2-model-gguf/commit/9549d06293f68c654dd51fe9398872aa9a97dec9', commit_message='Upload model_1_a-q4_k_m.gguf with huggingface_hub', commit_description='', oid='9549d06293f68c654dd51fe9398872aa9a97dec9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jacoblb/lab2-model-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='jacoblb/lab2-model-gguf'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import upload_file

local_path = "/content/drive/MyDrive/Scalable ML/Lab2/model_1_a-q8_0.gguf"
repo_id = "jacoblb/lab2-model-gguf"
path_in_repo = "model_1_a-q8_0.gguf"

upload_file(
    path_or_fileobj=local_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../Lab2/model_1_a-q8_0.gguf:   0%|          |  559kB / 1.32GB            

CommitInfo(commit_url='https://huggingface.co/jacoblb/lab2-model-gguf/commit/164d80db8a5a0188348741f54d1a0594c98d207a', commit_message='Upload model_1_a-q8_0.gguf with huggingface_hub', commit_description='', oid='164d80db8a5a0188348741f54d1a0594c98d207a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jacoblb/lab2-model-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='jacoblb/lab2-model-gguf'), pr_revision=None, pr_num=None)

#Task 2

## Load Llama-3.2 1B modell

In [ ]:
#KLISTRA IN VÄRDEN FRÅN DEN BÄSTA CONFFIGEN FRÅN SHA
# Load model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Prepare model for PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0156,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)
print(model.print_trainable_parameters())

==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.0156.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.11.6 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
None


In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}
)

def apply_template(examples):
    messages = examples["conversations"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

dataset = load_dataset("mlabonne/FineTome-100k", split="train")
dataset = dataset.shuffle(seed=42)

# train/test
train_and_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_and_test["train"]
test_dataset  = train_and_test["test"]

train_dataset = train_dataset.map(apply_template, batched=True)
test_dataset  = test_dataset.map(apply_template, batched=True)

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.


README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

##Training

In [ ]:
#KLISTRA IN VÄRDEN FRÅN DEN BÄSTA CONFFIGEN FRÅN SHA
PATH = "/content/drive/MyDrive/Scalable ML/Lab2/checkpoints_2"
small_test = test_dataset.shuffle(seed=42).select(range(500))

trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset = small_test,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=5e-05,
        lr_scheduler_type="linear",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        #num_train_epochs=1,
        max_steps = 6440,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.0366,
        warmup_steps=0,
        output_dir= PATH,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        seed=0,
        report_to = "none",
        remove_unused_columns = False,
    ),
)

try:
  trainer.train(resume_from_checkpoint=True)

except ValueError:
  trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/90000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90,000 | Num Epochs = 1 | Total steps = 6,440
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
3501,0.856100
3502,0.779000
3503,1.015100
3504,1.105100
3505,0.714200
3506,0.692200
3507,0.938900
3508,0.795800
3509,1.317900
3510,1.099800


Unsloth: Will smartly offload gradients to save VRAM!


##Evaluate

In [ ]:
metrics = trainer.evaluate()
print(metrics)

Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


{'eval_loss': 0.9175086617469788, 'eval_runtime': 127.0243, 'eval_samples_per_second': 3.936, 'eval_steps_per_second': 0.496, 'epoch': 0.2862222222222222}


##Save the lora model

In [ ]:
model.save_pretrained("/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2")

('/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2/tokenizer_config.json',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2/special_tokens_map.json',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2/chat_template.jinja',
 '/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2/tokenizer.json')

##Load the saved model for inference

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "/content/drive/MyDrive/Scalable ML/Lab2/lora_model_2", # YOUR MODEL YOU USED FOR TRAINING
      max_seq_length = max_seq_length,
      load_in_4bit=True,
      dtype=None,
  )
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"from": "human", "value": "Is Sweden a country?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True)

==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Unsloth 2025.11.6 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
Is Sweden a country?<|im_end|>
<|im_start|>assistant
Yes, Sweden is a country. It is a sovereign state located in Northern Europe. It is bordered by Norway to the north, Finland to the east, the Baltic Sea to the west, and the North Sea to the south. The capital and largest city is Stockholm. Sweden is a constitutional monarchy, meaning that the king is the head of state, but the prime minister is the head of government. Sweden is a member of the European Union (EU), the United Nations (UN), the NATO, and the OECD. The country is known for its high quality of life, liberal social policies, and strong economy.<|im_end|>


## Save a merged model

In [ ]:
model.save_pretrained_merged("/content/drive/MyDrive/Scalable ML/Lab2/model_2", tokenizer, save_method="merged_16bit")

config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:42<00:00, 42.77s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:05<00:00, 65.03s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Scalable ML/Lab2/model_2`


## Convert to gguf

The converting of hf to gguf can be found in "convert.ipynb".

## Upload the model to HuggingFace

In [2]:
!pip install -U "huggingface_hub"
from huggingface_hub import login
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.9/520.9 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.2 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.2.1 which is incompatible.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from huggingface_hub import upload_file

local_path = "/content/drive/MyDrive/Scalable ML/Lab2/model_2-q4_k_m.gguf"
repo_id = "jacoblb/lab2-model-gguf"
path_in_repo = "model_2-q4_k_m.gguf"

upload_file(
    path_or_fileobj=local_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../Lab2/model_2-q4_k_m.gguf:   3%|3         | 25.1MB /  808MB            

CommitInfo(commit_url='https://huggingface.co/jacoblb/lab2-model-gguf/commit/7d8192ef85ec98b6f5faf7ca7e0bb4d814c2bc5b', commit_message='Upload model_2-q4_k_m.gguf with huggingface_hub', commit_description='', oid='7d8192ef85ec98b6f5faf7ca7e0bb4d814c2bc5b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jacoblb/lab2-model-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='jacoblb/lab2-model-gguf'), pr_revision=None, pr_num=None)